# Agent 4 — Social Media Trend AI Agent
**Owner:** AI/ML Engineer (Member 3)

**Goal:** Analyze social media (Twitter) data to detect trending topics and public
sentiment relevant to a business category, using text preprocessing (NLTK) +
**LDA topic modeling** to surface hidden themes in the conversation.

**Dataset used:** `data/twitter_sentiment.csv` (Twitter Airline Sentiment Dataset)

**Output:** Trending topics, public interest/sentiment score, content recommendation
— returned by `agent_social(keyword)`.

> The shipped dataset is airline-focused, so in this demo "trends" = themes and
> sentiment around airline service. The same pipeline applies once you swap in
> tweets scraped for a specific business idea.


## 1. Imports & load data

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

DATA_DIR = "../../data"

tweets = pd.read_csv(f"{DATA_DIR}/twitter_sentiment.csv")
tweets = tweets.dropna(subset=["text"]).copy()
print(tweets.shape)
tweets[["airline", "airline_sentiment", "text"]].head()


## 2. Text preprocessing (NLTK-style cleaning)

In [ ]:
STOPWORDS = set([
    "the","a","an","and","or","but","is","are","was","were","to","of","in","on",
    "for","with","this","that","it","i","you","your","my","me","at","be","as",
    "have","has","had","not","so","just","its","im","amp","co","rt"
])

def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"@\w+", "", text)          # remove mentions
    text = re.sub(r"http\S+", "", text)        # remove links
    text = re.sub(r"[^a-z\s]", " ", text)      # keep letters only
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

tweets["clean_text"] = tweets["text"].apply(clean_tweet)
tweets[["text", "clean_text"]].head()


## 3. LDA topic modeling
Fit an LDA model over the cleaned tweet text to discover latent topics (themes)
in the conversation. Each topic is summarized by its top keywords.

In [ ]:
vectorizer = CountVectorizer(max_features=1000, min_df=5)
doc_term_matrix = vectorizer.fit_transform(tweets["clean_text"])

N_TOPICS = 6
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=10)
lda.fit(doc_term_matrix)

feature_names = vectorizer.get_feature_names_out()

def get_topic_keywords(model, feature_names, top_n=8):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_terms = [feature_names[i] for i in topic.argsort()[::-1][:top_n]]
        topics.append({"topic_id": topic_idx, "keywords": top_terms})
    return topics

topics = get_topic_keywords(lda, feature_names)
for t in topics:
    print(t)


## 4. Sentiment & interest score per airline
Aggregate sentiment distribution per airline as a proxy for "public interest score"
for that brand/topic.

In [ ]:
sentiment_by_airline = (
    tweets.groupby("airline")["airline_sentiment"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
    .round(3)
)
sentiment_by_airline


## 5. `agent_social(keyword)`

In [ ]:
def interest_score(subset):
    """Simple 0-100 'public interest' score from volume + positive sentiment ratio."""
    if len(subset) == 0:
        return 0.0
    pos_ratio = (subset["airline_sentiment"] == "positive").mean()
    volume_score = min(len(subset) / len(tweets), 1.0) * 50
    return round(pos_ratio * 50 + volume_score, 2)


def agent_social(keyword: str, top_n_topics: int = 3) -> dict:
    """Social Media Trend AI Agent.

    Input : business category keyword, e.g. 'delivery service'
    Output: trending topics, public interest score, content recommendation
    """
    keyword_lc = keyword.lower()
    mask = tweets["clean_text"].str.contains(keyword_lc.split()[0], na=False)
    subset = tweets[mask] if mask.any() else tweets

    score = interest_score(subset)
    sentiment_dist = subset["airline_sentiment"].value_counts(normalize=True).round(3).to_dict()

    return {
        "agent": "Social Media Trend Agent",
        "input_keyword": keyword,
        "num_posts_analyzed": int(len(subset)),
        "public_interest_score": score,
        "sentiment_distribution": sentiment_dist,
        "trending_topics": topics[:top_n_topics],
        "recommendation": (
            f"Public sentiment is mostly '{max(sentiment_dist, key=sentiment_dist.get)}'. "
            f"Top discussion themes: {', '.join(topics[0]['keywords'][:4])}."
        ),
    }


## 6. Test the agent

In [ ]:
import json
print(json.dumps(agent_social("delivery"), indent=2))


## 7. Next steps
- Move `clean_tweet`, the fitted `vectorizer`/`lda`, `get_topic_keywords` and
  `agent_social()` into `agent4_social/topic_modeling.py` and
  `agent4_social/social_agent.py`.
- Cache the fitted `CountVectorizer` and `LatentDirichletAllocation` models
  (joblib) at startup rather than retraining per request.
- For a real "social media trend" use case, replace the airline tweets with
  tweets scraped/streamed for the user's specific business category.
